## Hyde Implementation

In [ ]:
# pip install rank-bm25 sentence-transformers chromadb numpy
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
import numpy as np
import chromadb
# ── Your document corpus ─────────────────────────────────────
chunks = [
"Students must submit CNIC and transcripts for admission to KUST.",
"The semester fee is PKR 85,000 payable in two installments.",
"Late payment of fees incurs a 5% penalty after the due date.",
"Article 7.3.b states that refunds must be processed within 30 days.",
"The library is open Monday to Friday from 8am to 10pm.",
"CNIC verification is mandatory before exam registration.",
]
chunk_ids = list(range(len(chunks)))
# ── Build BM25 index (sparse retrieval) ─────────────────────
tokenised = [chunk.lower().split() for chunk in chunks]
bm25 = BM25Okapi(tokenised)
# ── Build dense index (ChromaDB) ─────────────────────────────
embedder = SentenceTransformer("all-MiniLM-L6-v2")
client = chromadb.Client()
collection = client.create_collection("hybrid_demo", metadata={"hnsw:space": "cosine"})
embeddings = embedder.encode(chunks, normalize_embeddings=True).tolist()
collection.add(ids=[str(i) for i in chunk_ids],
embeddings=embeddings,
documents=chunks)
# ── Reciprocal Rank Fusion ───────────────────────────────────0
def reciprocal_rank_fusion(ranked_lists, k=60):
   scores = {}
   for ranked_list in ranked_lists:
       for rank, doc_id in enumerate(ranked_list, start=1):
          scores[doc_id] = scores.get(doc_id, 0) + 1.0 / (k + rank)
   return sorted(scores.items(), key=lambda x: x[1], reverse=True)
# ── Hybrid search function ────────────────────────────────────
def hybrid_search(query, top_k=3, bm25_k=10, dense_k=10):
# Step 1: BM25 sparse retrieval
    query_tokens = query.lower().split()
    bm25_scores = bm25.get_scores(query_tokens)
    bm25_ranked = np.argsort(bm25_scores)[::-1][:bm25_k].tolist()
# Step 2: Dense retrieval
    q_emb = embedder.encode([query], normalize_embeddings=True).tolist()
    dense_result = collection.query(query_embeddings=q_emb, n_results=dense_k)
    dense_ranked = [int(i) for i in dense_result["ids"][0]]
    print(f"BM25 top-3: {[chunks[i][:40] for i in bm25_ranked[:3]]}")
    print(f"Dense top-3: {[chunks[i][:40] for i in dense_ranked[:3]]}")
# Step 3: Reciprocal Rank Fusion
    fused = reciprocal_rank_fusion([bm25_ranked, dense_ranked])
    top_ids = [doc_id for doc_id, score in fused[:top_k]]
    return [{"id": i, "text": chunks[i], "rrf_score": dict(fused)[i]} for i in top_ids]
# ── Test queries ──────────────────────────────────────────────
print("=== Query 1: semantic (dense should win) ===")
results = hybrid_search("what are the payment deadlines?")
for r in results:
  print(f" [{r['rrf_score']:.4f}] {r['text'][:60]}...")
  
print("\n=== Query 2: exact term (BM25 should win) ===")

results = hybrid_search("Article 7.3.b refund")
for r in results:
  print(f" [{r['rrf_score']:.4f}] {r['text'][:60]}...")
# BM25 finds "Article 7.3.b" by exact match that dense might miss

/home/madiha/My Notes/2026/Code/RAG (basic to advance)/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2308.23it/s]


=== Query 1: semantic (dense should win) ===
BM25 top-3: ['Late payment of fees incurs a 5% penalty', 'CNIC verification is mandatory before ex', 'The library is open Monday to Friday fro']
Dense top-3: ['Late payment of fees incurs a 5% penalty', 'Article 7.3.b states that refunds must b', 'The semester fee is PKR 85,000 payable i']
 [0.0328] Late payment of fees incurs a 5% penalty after the due date....
 [0.0318] Article 7.3.b states that refunds must be processed within 3...
 [0.0315] CNIC verification is mandatory before exam registration....

=== Query 2: exact term (BM25 should win) ===
BM25 top-3: ['Article 7.3.b states that refunds must b', 'CNIC verification is mandatory before ex', 'The library is open Monday to Friday fro']
Dense top-3: ['Article 7.3.b states that refunds must b', 'Late payment of fees incurs a 5% penalty', 'The semester fee is PKR 85,000 payable i']
 [0.0328] Article 7.3.b states that refunds must be processed within 3...
 [0.0318] Late payment of fees inc

## ms-marco-MiniLM cross-encoder

In [ ]:
# pip install sentence-transformers
from sentence_transformers import CrossEncoder
import numpy as np
# Load the cross-encoder reranker
# ms-marco-MiniLM-L-6-v2: fast, small, excellent for English retrieval
# ms-marco-MiniLM-L-12-v2: slightly better quality, slower
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
query = "What is the penalty for paying fees late?"
# Stage 1: initial retrieval candidates (simulate with a list)
candidates = [
"The semester fee is PKR 85,000 payable in two installments.",
"Late payment of fees incurs a 5% penalty after the due date.",
"Students must submit CNIC and transcripts for admission.",
"The library is open Monday to Friday from 8am to 10pm.",
"CNIC verification is mandatory before exam registration.",
]
# Stage 2: rerank
# CrossEncoder.predict() takes pairs of (query, document) and returns a score
pairs = [(query, doc) for doc in candidates]
scores = reranker.predict(pairs)
print("Reranker scores (higher = more relevant):")
for doc, score in sorted(zip(candidates, scores), key=lambda x: x[1], reverse=True):
 print(f" {score:7.3f} | {doc[:65]}...")
# Expected: "Late payment of fees incurs a 5% penalty" gets highest score
# "Library hours" gets near-zero score -- cross-encoder correctly ignores it
# ── Full two-stage pipeline ───────────────────────────────────
def retrieve_and_rerank(query, vector_results, top_k_final=3):
 if not vector_results:
   return []
# Score all candidates with cross-encoder
pairs = [(query, doc["text"]) for doc in vector_results]
scores = reranker.predict(pairs)
# Sort by reranker score
reranked = sorted(zip(vector_results, scores),
key=lambda x: x[1], reverse=True)
return [{"text": doc["text"], "source": doc.get("source",""), "rerank_score": float(score)}
     for doc, score in reranked[:top_k_final]]

/home/madiha/My Notes/2026/Code/RAG (basic to advance)/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 105/105 [00:00<00:00, 6060.80it/s]


Reranker scores (higher = more relevant):
   8.900 | Late payment of fees incurs a 5% penalty after the due date....
  -8.855 | The semester fee is PKR 85,000 payable in two installments....
 -11.183 | The library is open Monday to Friday from 8am to 10pm....
 -11.265 | CNIC verification is mandatory before exam registration....
 -11.335 | Students must submit CNIC and transcripts for admission....


NameError: name 'vector_results' is not defined

## Cohere Rerank API

In [ ]:
# pip install cohere
import cohere
# Cohere rerank is API-based -- requires a Cohere API key
# Free tier available at cohere.com
co = cohere.Client("your-cohere-api-key")
query = "What is the penalty for paying fees late?"
documents = [
"The semester fee is PKR 85,000 payable in two installments.",
"Late payment of fees incurs a 5% penalty after the due date.",
"Students must submit CNIC and transcripts for admission.",
"The library is open Monday to Friday from 8am to 10pm.",
]
results = co.rerank(
query=query,
documents=documents,
top_n=3,
model="rerank-multilingual-v3.0"
)
# supports Urdu and other languages
print("Cohere reranked results:")
for item in results.results:
  print(f" rank={item.index+1} | score={item.relevance_score:.4f}")
  print(f" text: {documents[item.index][:60]}...")
# Cohere rerank-multilingual-v3.0 supports Urdu
# Useful for bilingual RAG systems like your KUST chatbot
# Cost: small per call, much lower than LLM calls

## HyDE implementation

In [ ]:
# HyDE implementation
# pip install openai sentence-transformers chromadb
from openai import OpenAI
from sentence_transformers import SentenceTransformer
openai_client = OpenAI()
# requires OPENAI_API_KEY env variable
embedder = SentenceTransformer("all-MiniLM-L6-v2")
def generate_hypothetical_document(query):
   response = openai_client.chat.completions.create(
   model="gpt-3.5-turbo",
   temperature=0.7,
# slight creativity for diversity
   max_tokens=150,
   messages=[{
"role": "system",
"content": ("Write a short factual passage (2-3 sentences) that "
"directly answers the following question. Write as if "
"this is an excerpt from an official university policy "
"document. Be specific and factual.")
}, {
"role": "user",
"content": f"Question: {query}"
}]
)
   return response.choices[0].message.content
def hyde_embed(query):
# Generate hypothetical document
     hyp_doc = generate_hypothetical_document(query)
     print(f"Hypothetical document: {hyp_doc}")
# Embed the HYPOTHETICAL DOCUMENT instead of the raw query
     hyp_embedding = embedder.encode([hyp_doc], normalize_embeddings=True)
     return hyp_embedding, hyp_doc
# Compare: raw query embedding vs HyDE embedding
query = "fees penalty"
raw_emb = embedder.encode([query], normalize_embeddings=True)
hyde_emb, hyp_doc = hyde_embed(query)
# Both embeddings can be used for vector search
# In practice: run retrieval with hyde_emb
# The retrieved chunks should be more precise than with raw_emb
# To run the comparison on your ChromaDB collection:
# raw_results = collection.query(query_embeddings=raw_emb.tolist(), n_results=3)
# hyde_results = collection.query(query_embeddings=hyde_emb.tolist(), n_results=3)
# Print both and compare -- HyDE should surface more relevant chunks
# Cost note: HyDE adds one LLM call per query at retrieval time.
# Worth it for short or ambiguous queries.
# Not worth it for long, detailed queries that already embed well.

## Multi-Query Expansion

In [ ]:
from openai import OpenAI
import json
client = OpenAI()
def generate_query_variants(original_query, n=4):
  response = client.chat.completions.create(
model="gpt-3.5-turbo",
temperature=0.8,
messages=[{
"role": "system",
"content": (
f"Generate {n} different rephrasings of the following question "
"to maximise retrieval recall from a document database. "
"Each rephrasing should use different words but ask the same thing. "
f"Return ONLY a JSON array of {n} strings, nothing else."
)
}, {
"role": "user",
"content": original_query
}]
)
try:
  variants = json.loads(response.choices[0].message.content)
  return [original_query] + variants      # include original
except:
  return [original_query] # fallback to original if parsing fails
def multi_query_retrieve(query, collection, embedder, k=3):
 variants = generate_query_variants(query, n=3)
 print(f"Query variants generated:")
 for v in variants:
   print(f" - {v}")
# Retrieve for each variant
 all_ranked_lists = []
 seen_texts = {}
 for variant in variants:
  emb = embedder.encode([variant], normalize_embeddings=True).tolist()
  results = collection.query(query_embeddings=emb, n_results=k)
  ids_in_order = results["ids"][0]
  all_ranked_lists.append(ids_in_order)
  for doc_id, text in zip(results["ids"][0], results["documents"][0]):
     seen_texts[doc_id] = text
# Merge with RRF
rrf_scores = {}
for ranked_list in all_ranked_lists:
 for rank, doc_id in enumerate(ranked_list, 1):
    rrf_scores[doc_id] = rrf_scores.get(doc_id, 0) + 1.0 / (60 + rank)
merged = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)
return [{"id": doc_id, "text": seen_texts[doc_id], "score": score}
for doc_id, score in merged[:k]]

## Query Decomposition

In [ ]:
from openai import OpenAI
import json
client = OpenAI()
def decompose_query(complex_query):
  response = client.chat.completions.create(
model="gpt-3.5-turbo",
temperature=0,
messages=[{
"role": "system",
"content": (
"Break the following complex question into 2-4 simpler sub-questions "
"that can each be answered by searching a document database independently. "
"Return ONLY a JSON array of sub-question strings."
)
}, {
"role": "user",
"content": complex_query
}]
)
  return json.loads(response.choices[0].message.content)
def decomposed_rag(complex_query, retrieve_fn, generate_fn):
# Step 1: decompose
  sub_questions = decompose_query(complex_query)
  print(f"Decomposed into: {sub_questions}")
# Step 2: retrieve and answer each sub-question
  sub_answers = []
  all_sources = []
  for sq in sub_questions:
     chunks = retrieve_fn(sq)
     answer = generate_fn(sq, chunks)
     sub_answers.append(f"Q: {sq} A: {answer}")
     all_sources.extend([c.get("source","") for c in chunks])
# Step 3: synthesise final answer from sub-answers
  synthesis_prompt = ("Using the following question-answer pairs as context, " f"answer the original question: {complex_query}" + "".join(sub_answers)
)
  final_answer = generate_fn(complex_query, [], custom_context=synthesis_prompt)
  return {"answer": final_answer, "sources": list(set(all_sources))}

## Contextual Compression

In [ ]:
# LangChain contextual compression with LLM extractor
# pip install langchain langchain-openai

from langchain.retrievers import ContextualCompressionRetriever
from langchain.retrievers.document_compressors import LLMChainExtractor
from langchain_openai import ChatOpenAI

# The compressor uses an LLM to extract relevant sentences
llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)
compressor = LLMChainExtractor.from_llm(llm)

# Wrap your base retriever with the compression layer
# base_retriever = your ChromaDB or FAISS retriever from Day 4
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=base_retriever  # your existing retriever
)

# Use exactly like any other retriever
query = "What is the penalty for late fee payment?"
compressed_docs = compression_retriever.get_relevant_documents(query)

print("Compressed results:")
for doc in compressed_docs:
    print(f" Source: {doc.metadata.get('source', 'unknown')}")
    print(f" Compressed text: {doc.page_content}")
    print(f" Length: {len(doc.page_content.split())} words")

# Without compression: might return 400-token chunk
# With compression: returns only "Late payment incurs a 5% penalty after the due date."


# ── Lightweight alternative: sentence-level extraction ────────
# If you want to avoid the extra LLM call cost of LLMChainExtractor:
from sentence_transformers import SentenceTransformer, util
import nltk
nltk.download("punkt", quiet=True)
from nltk.tokenize import sent_tokenize

sent_model = SentenceTransformer("all-MiniLM-L6-v2")

def extract_relevant_sentences(query, chunk_text, threshold=0.4):
    sentences = sent_tokenize(chunk_text)
    if len(sentences) <= 2:
        return chunk_text  # too short to compress
        
    q_emb = sent_model.encode(query, normalize_embeddings=True)
    s_emb = sent_model.encode(sentences, normalize_embeddings=True)
    
    similarities = util.cos_sim(q_emb, s_emb)[0]
    relevant = [
        sent for sent, sim in zip(sentences, similarities)
        if sim.item() > threshold
    ]
    
    return " ".join(relevant) if relevant else chunk_text

chunk = (
    "The semester fee is PKR 85,000 payable in two installments. "
    "Late payment of fees incurs a 5% penalty after the due date. "
    "The library is open from 8am to 10pm. "
    "CNIC verification is required before exam registration."
)

compressed = extract_relevant_sentences("what is the fee penalty?", chunk)

print(f"Original: {len(chunk.split())} words")
print(f"Compressed: {len(compressed.split())} words")
print(f"Result: {compressed}")